# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [11]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

C:\Users\Personal\AppData\Roaming\Python\Python311\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
<frozen importlib._bootstrap_external>:729: ResourceWarning: unclosed <socket.socket fd=1820, family=23, type=1, proto=0, laddr=('::1', 55161, 0, 0), raddr=('::1', 8080, 0, 0)>


In [13]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [14]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [15]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [16]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

In [17]:
MAX_PASSAGES = 500

passages = passages[:MAX_PASSAGES]

embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

C:\Users\Personal\anaconda3\Lib\site-packages\ipywidgets\widgets\widget.py:528: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [18]:
print(embeddings.shape, embeddings.dtype)

(500, 768) float32


In [19]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

In [20]:
!pip install faiss-cpu


C:\Users\Personal\anaconda3\Lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  return process_handler(cmd, _system_body)
C:\Users\Personal\anaconda3\Lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  return process_handler(cmd, _system_body)
C:\Users\Personal\anaconda3\Lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  return process_handler(cmd, _system_body)


In [21]:
import faiss
import numpy as np

# Dimensión de los embeddings
D = embeddings.shape[1]

# Índice FAISS (L2; con embeddings normalizados equivale a cosine)
index = faiss.IndexFlatL2(D)

# Agregar embeddings
index.add(embeddings)

print("Total de vectores en el índice:", index.ntotal)


Total de vectores en el índice: 500


<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [23]:
# Búsqueda
k = 10
Dists, Ids = index.search(query_vec, k)

results = []
for rank, idx in enumerate(Ids[0]):
    results.append({
        "rank": rank + 1,
        "chunk_id": int(idx),
        "distance": float(Dists[0][rank]),
        "text": chunks_df.iloc[idx]["text"]
    })

results


[{'rank': 1,
  'chunk_id': 1,
  'distance': 0.27639898657798767,
  'text': "Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a bad battery when in reality it indicates a problem with the vehicle's charging system. Alternatively,"},
 {'rank': 2,
  'chunk_id': 5,
  'distance': 0.33675554394721985,
  'text': 'otective diodes cannot be used, a battery wi

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [24]:
!pip install qdrant-client


In [25]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, Filter, FieldCondition, MatchValue

In [26]:
client = QdrantClient(location=":memory:")

In [27]:
client.recreate_collection(
    collection_name="docs",
    vectors_config=VectorParams(
        size=embeddings.shape[1],  # 768
        distance=Distance.COSINE
    )
)

C:\Users\Personal\AppData\Local\Temp\ipykernel_17888\924002763.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [28]:
points = []

for i in range(len(embeddings)):
    point = PointStruct(
        id=i,
        vector=embeddings[i],
        payload={
            "text": chunks_df.iloc[i]["text"],
            "doc_id": int(chunks_df.iloc[i]["doc_id"])
        }
    )
    points.append(point)

In [29]:
client.upsert(
    collection_name="docs",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [30]:
search_result = client.query_points(
    collection_name="docs",
    query=query_vec[0],
    limit=5
)

In [31]:
print("--- Top Resultados Qdrant ---")

for i, point in enumerate(search_result.points):
    print(f"#{i+1} [{point.score:.4f}] {point.payload['text'][:120]}...")

--- Top Resultados Qdrant ---
#1 [0.8618] Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
#2 [0.8316] otective diodes cannot be used, a battery will simply destroy the diodes and damage itself. An ESR meter known not to ha...
#3 [0.8222] ad battery when in reality it indicates a problem with the vehicle's charging system. Alternatively, an ammeter may be f...
#4 [0.8174] Capacity loss Capacity loss or capacity fading is a phenomenon observed in rechargeable battery usage where the amount o...
#5 [0.8081] increase; in many cases the EMF remains more or less constant during most of the discharge, with the voltage drop across...


## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [32]:
!pip install pymilvus

In [33]:
from pymilvus import connections
connections.connect(host="localhost", port="19530")
print("Milvus conectado correctamente")


Milvus conectado correctamente


In [34]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection

connections.connect("default", host="localhost", port="19530")

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=D),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=2048)
]

schema = CollectionSchema(fields, description="Wikipedia chunks")
collection = Collection("wiki_chunks_milvus", schema)




In [36]:
# Cantidad de embeddings que sí existen
n = len(embeddings)

# Tomar SOLO la última parte de los textos
texts = chunks_df["text"].iloc[-n:].tolist()

# Crear IDs únicos para esta tanda (continúan después de lo anterior)
start_id = collection.num_entities
ids = list(range(start_id, start_id + n))

# Insertar en Milvus
collection.insert([ids, embeddings.tolist(), texts])
collection.flush()

print("Insertados:", n)
print("Total en Milvus:", collection.num_entities)


Insertados: 500
Total en Milvus: 500


In [37]:
index_params = {
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "params": {"M": 16, "efConstruction": 200}
}

collection.create_index("embedding", index_params)
collection.load()


In [38]:
def milvus_search(query_embedding, k=5):
    search_params = {"metric_type": "COSINE", "params": {"ef": 64}}

    results = collection.search(
        data=query_embedding.tolist(),
        anns_field="embedding",
        param=search_params,
        limit=k,
        output_fields=["text"]
    )

    output = []
    for hit in results[0]:
        output.append((
            hit.id,
            hit.score,
            hit.entity.get("text")
        ))
    return output


In [39]:
milvus_search(query_vec, k=5)


[(1,
  0.8618005514144897,
  "ed under Australian Treasury Guidelines for electronic commerce and the Australian Competition and Consumer Commission regulates and offers advice on how to deal with businesses online, and offers specific advice on what happens if things go wrong. In the United Kingdom, The Financial Services Authority (FSA) was formerly the regulating authority for most aspects of the EU's Payment Services Directive (PSD), until its replacement in 2013 by the Prudential Regulation Authority and the Financial Conduct Authority. The UK implemented the PSD through the Payment Services Regulations 2009 (PSRs), which came into effect on 1 November 2009. The PSR affects firms providing payment services and their customers. These firms include banks, non-bank credit card issuers and non-bank merchant acquirers,"),
 (5,
  0.8316222429275513,
  "ta integrity and security are pressing issues for electronic commerce. Aside from traditional e-commerce, the terms m-Commerce (mobile c

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [1]:
pip install weaviate-client


Note: you may need to restart the kernel to use updated packages.


In [22]:
import weaviate
from weaviate.connect import ConnectionParams

client = weaviate.WeaviateClient(
    connection_params=ConnectionParams.from_url(
        "http://localhost:8080",
        grpc_port=50051
    )
)

client.connect()
client.is_ready()



True

In [23]:
from weaviate.classes.config import Configure, Property, DataType

# Borra todo (equivalente a delete_all)
client.collections.delete_all()

# Crea la colección Document
client.collections.create(
    name="Document",
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="doc_id", data_type=DataType.INT),
        Property(name="chunk_id", data_type=DataType.INT),
    ]
)


C:\Users\Personal\AppData\Roaming\Python\Python311\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


In [26]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

MAX_CHUNKS = 500 

# Recorta el dataframe
chunks_df = chunks_df.iloc[:MAX_CHUNKS].reset_index(drop=True)

texts = chunks_df["text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=32
)

print("Chunks usados:", len(chunks_df))
print("Embeddings:", embeddings.shape)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Chunks usados: 500
Embeddings: (500, 384)


In [27]:
assert len(chunks_df) == embeddings.shape[0]


In [29]:
client.collections.delete("Document")


In [30]:
client.collections.create(
    name="Document",
    vectorizer_config=None  # porque tú envías los embeddings
)


In [31]:
collection = client.collections.get("Document")

with collection.batch.dynamic() as batch:
    for i, row in chunks_df.iterrows():
        batch.add_object(
            properties={
                "text": row["text"],
                "doc_id": int(row["doc_id"]),
                "chunk_id": int(row["chunk_id"])
            },
            vector=embeddings[i].tolist()
        )


In [36]:
def weaviate_search(query_embedding, k=5):
    collection = client.collections.get("Document")

    res = collection.query.near_vector(
        near_vector=query_embedding[0].tolist(),
        limit=k,
        return_metadata=["distance"]
    )

    hits = []
    for i, obj in enumerate(res.objects):
        hits.append((
            i,
            obj.metadata.distance,
            obj.properties["text"],
            obj.properties
        ))

    return hits


## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [40]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(
    Settings(persist_directory="./chroma_db")
)

collection = client.create_collection(name="wiki_chunks")


In [41]:
collection.add(
    ids=[str(i) for i in range(len(embeddings))],
    embeddings=embeddings.tolist(),
    documents=chunks_df["text"].tolist(),
    metadatas=[
        {"doc_id": int(r.doc_id), "chunk_id": int(r.chunk_id)}
        for r in chunks_df.itertuples()
    ]
)


In [42]:
def chroma_search(query_embedding, k=5):
    res = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k
    )
    return res


In [45]:
client.delete_collection("wiki_chunks")


In [47]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(Settings(persist_directory="./chroma_db"))

# Si ya existe, elimínala primero
try:
    client.delete_collection("wiki_chunks")
except:
    pass

# Crear colección vacía
collection = client.create_collection(name="wiki_chunks")


In [48]:
collection.add(
    embeddings=[query_vec[0].tolist()],  # tu vector de 768 dim
    metadatas=[{"source": "wiki"}],
    documents=["Texto de ejemplo"],
    ids=["doc1"]
)


In [49]:
chroma_search(query_vec, k=5)


{'ids': [['doc1']],
 'embeddings': None,
 'documents': [['Texto de ejemplo']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'wiki'}]],
 'distances': [[0.0]]}

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?
